<a href="https://colab.research.google.com/github/hemel5612-lgtm/Aleurodicus-rugioperculatus/blob/main/Aleurodicus_rugioperculatus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# STEP 0 — Lock the analytical template and audit the workbook
# ============================================================

# Optional installs for later steps
# !pip -q install scikit-posthocs factor_analyzer openpyxl statsmodels

from google.colab import drive
drive.mount('/content/drive')

import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

# -----------------------------
# Paths
# -----------------------------
BASE_DIR = Path("/content/drive/MyDrive/Data Analysis March 2026")
INPUT_XLSX = BASE_DIR / "Aleurodicus_rugioperculatus.xlsx"
OUTPUT_DIR = BASE_DIR / "Mishu"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_XLSX.exists():
    raise FileNotFoundError(
        f"Input workbook not found at:\n{INPUT_XLSX}\n\n"
        "Upload/copy Aleurodicus_rugioperculatus.xlsx into this folder first."
    )

# -----------------------------
# Analysis template locked from PDF
# -----------------------------
TEMPLATE = {
    "correction_formula": "Abbott corrected mortality",
    "time_points": ["24 HAS", "48 HAS", "72 HAS"],
    "life_stages": ["Egg", "Nymph", "Adult"],
    "assumption_tests": ["Shapiro-Wilk", "Levene"],
    "group_comparison": "Kruskal-Wallis + Conover-Iman post-hoc + Holm-Bonferroni letters",
    "factorial_model": "Aligned Rank Transform (ART) ANOVA",
    "dose_response": "LL.4 log-logistic at 72 HAS; fallback LL.3 if lower asymptote is unstable",
    "toxicity_outputs": ["LC50", "LC90", "95% CI", "Slope ± SE", "Chi-square goodness-of-fit"],
    "multivariate": "PCA at 72 HAS with KMO",
    "reporting_style": "Mean ± SE plus grouping letters; publication-style tables and figures"
}

print("STEP 0 COMPLETE: PDF-based analysis template locked.\n")
for k, v in TEMPLATE.items():
    print(f"{k}: {v}")

# -----------------------------
# Workbook audit
# -----------------------------
xlsx = pd.ExcelFile(INPUT_XLSX)
print("\nSheets found:")
for s in xlsx.sheet_names:
    print(" -", s)

audit_rows = []

for sheet_name in xlsx.sheet_names:
    df = pd.read_excel(INPUT_XLSX, sheet_name=sheet_name)

    # Fix merged-like concentration labels
    if "Conc" in df.columns:
        df["Conc"] = df["Conc"].ffill()

    control_mask = df["Conc"].astype(str).str.strip().str.lower().eq("control")
    treatment_df = df.loc[~control_mask].copy()

    audit_rows.append({
        "Sheet": sheet_name,
        "Total rows": len(df),
        "Control reps": int(control_mask.sum()),
        "Treatment levels": int(treatment_df["Conc"].nunique()),
        "Rows per level": str(df.groupby("Conc").size().to_dict()),
        "Missing cells after Conc fill": int(df.isna().sum().sum())
    })

audit_df = pd.DataFrame(audit_rows)
print("\nWorkbook audit:")
display(audit_df)

# -----------------------------
# Critical dataset notes
# -----------------------------
print("\nCritical notes:")
print("1. Dataset structure is baseline + 24/48/72 hour counts for Egg, Nymph, Adult.")
print("2. This is a laboratory bioassay dataset, not a 1/3/7/15 DAS field dataset.")
print("3. Ravzoom 14.5 SC requires forward-fill of concentration labels before analysis.")
print(f"4. Output folder is ready: {OUTPUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: Input workbook not found at:
/content/drive/MyDrive/Data Analysis March 2026/Aleurodicus_rugioperculatus.xlsx

Upload/copy Aleurodicus_rugioperculatus.xlsx into this folder first.